# 1. Задача и первичная диагностика

**Контекст.** Аналитик автодилерского холдинга готовит данные о продажах автомобилей в Казахстане за январь–сентябрь 2019 года для руководства «Меркур Авто».

**Техническая цель:** получить очищенный датасет и воспроизводимый скрипт обработки. **Аналитическая цель:** описать рынок, оценить долю и структуру продаж «Меркур Авто», определить обоснованные направления дальнейшей проверки и роста.

Источник — предоставленный файл `data/autokz2019_raw.csv`, копия исходника без изменений. Ссылка задания: https://code.s3.yandex.net/high-education_higher-data-analytics/autokz2019.csv . Интернет-версия не подменяет пользовательский файл.

**Ключевые ограничения:** нет уникального идентификатора сделки, данных о марже, расходах, запасах, лидах и целевых KPI. Количество в строке может превышать единицу. «Компания» содержит разные типы участников, поэтому рейтинг компаний — сравнение записей источника, а не доказанный рейтинг сопоставимых дилерских центров. Все рыночные доли относятся к охвату этого датасета.

In [1]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
from IPython.display import display

# Тетради можно открывать из папки проекта или из coursework.
ROOT = Path.cwd() if Path('clean_data.py').exists() else Path.cwd() / 'coursework'
sys.path.insert(0, str(ROOT))
import os
os.chdir(ROOT)
from clean_data import load_raw, normalize, aggregate_clean, read_clean, run, validate, CATEGORIES, DROP, COMPANIES, MONTHS
pd.set_option('display.max_columns', 12)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

raw = load_raw('data/autokz2019_raw.csv')
print(f'Строк: {len(raw):,}; столбцов: {raw.shape[1]}')
display(raw.head(5))

Строк: 39,966; столбцов: 25


,Год,Месяц,Компания,Бренд,Модель,Модификация,...,"Продажа, USD",Область,Сегментация 2013,Класс 2013,Сегментация Eng,Локализация производства
0,2019,Май,Mercur Auto,Audi,A3,TFSI,...,"28,115.00",г.Алматы,Легковые автомобили,C класс,C,Импорт
1,2019,Август,Mercur Auto,Audi,A3,TFSI,...,"32,246.99",г.Алматы,Легковые автомобили,C класс,C,Импорт
2,2019,Апрель,Mercur Auto,Audi,A4,TFSI,...,"32,000.00",г.Алматы,Легковые автомобили,D класс,D,Импорт
3,2019,Июль,Mercur Auto,Audi,A4,TFSI,...,"31,929.00",г.Алматы,Легковые автомобили,D класс,D,Импорт
4,2019,Июль,Mercur Auto,Audi,A4,TFSI,...,"31,929.00",г.Алматы,Легковые автомобили,D класс,D,Импорт


## Структура, пропуски и периоды
Пустая строка и неизвестное значение отдельного признака — разные случаи. Не удаляем все строки, в которых есть хотя бы один пропуск.

In [2]:
profile = pd.DataFrame({'type': raw.dtypes.astype(str), 'missing': raw.isna().sum(),
                        'missing_pct': raw.isna().mean() * 100, 'unique': raw.nunique()})
display(profile)
display(raw.groupby('Месяц').agg(rows=('Год', 'size'), units=('Количество', 'sum')).reindex(MONTHS).dropna())
print('Полностью пустых строк:', raw.isna().all(axis=1).sum())
print('Годы:', raw['Год'].unique())

,type,missing,missing_pct,unique
Год,int64,0,0.00,1
Месяц,str,0,0.00,9
Компания,str,0,0.00,40
Бренд,str,0,0.00,46
Модель,str,0,0.00,273
Модификация,str,3591,8.99,1383
Год выпуска,str,501,1.25,9
Страна-производитель,str,0,0.00,19
Вид топлива,str,3140,7.86,10
"Объём двиг, л,",str,4258,10.65,114


,rows,units
Месяц,,
Январь,"3,384.00","4,517.00"
Февраль,"3,137.00","4,493.00"
Март,"3,662.00","5,055.00"
Апрель,"4,177.00","5,868.00"
Май,"4,798.00","6,438.00"
Июнь,"4,754.00","6,459.00"
Июль,"5,930.00","7,778.00"
Август,"4,812.00","6,625.00"
Сентябрь,"5,312.00","6,641.00"


Полностью пустых строк: 0
Годы: [2019]


## Явные дубликаты и риск потери продаж
Удаляем полные повторы **по всем исходным полям** согласно заданию. Данные не позволяют отличить повторную загрузку от двух одинаковых сделок; поэтому далее обязательно сравниваем результаты с вариантом без удаления. Удаление столбцов до этой проверки дополнительно стирает различия между строками и не используется как основание для удаления продаж.

In [3]:
drop_ru = ['Наименование дилерского центра', 'Форма расчета', 'Сегмент', 'Сегментация Eng',
           'Локализация производства', 'Тип клиента', 'Модификация']
variants = {'Исходник': raw, 'Без полных повторов': raw.drop_duplicates(),
            'Если сначала удалить поля (не используем)': raw.drop(columns=drop_ru).drop_duplicates()}
display(pd.DataFrame({name: {'rows': len(z), 'units': z['Количество'].sum(), 'sales_usd': z['Продажа, USD'].sum()}
                      for name, z in variants.items()}).T)

,rows,units,sales_usd
Исходник,"39,966.00","53,874.00","1,349,158,968.06"
Без полных повторов,"21,268.00","35,165.00","815,179,587.19"
Если сначала удалить поля (не используем),"16,495.00","30,335.00","677,578,625.75"


## Проверка аномалий
Возврат и оптовая партия не являются автоматически ошибками. Причину или покупателя оптового заказа источник не раскрывает.

In [4]:
mask = raw['Количество'].isna() | raw['Количество'].lt(0) | raw['Количество'].ge(100)
display(raw.loc[mask, ['Компания', 'Бренд', 'Модель', 'Количество', 'Цена, USD', 'Продажа, USD']])
expected = raw['Количество'] * raw['Цена, USD']
known = expected.notna()
print('Максимальное расхождение цена × количество и сумма, USD:',
      (raw.loc[known, 'Продажа, USD'] - expected[known]).abs().max())

,Компания,Бренд,Модель,Количество,"Цена, USD","Продажа, USD"
16741,БИПЭК АВТО,Kia,Sportage,NaN,"20,700.00",0.00
16885,БИПЭК АВТО,Lada,4x4,115.00,"9,000.00","1,035,000.00"
17368,БИПЭК АВТО,Lada,Granta,NaN,"7,100.00",0.00
18676,БИПЭК АВТО,Lada,XRAY,NaN,"13,800.00",0.00
25555,БИПЭК АВТО,Skoda,Octavia,100.00,"18,700.00","1,870,000.00"
25728,БИПЭК АВТО,Skoda,Superb,-1.00,"35,588.25","-35,588.25"
37416,БИПЭК АВТО,UAZ,3909,NaN,"11,200.00",0.00
37727,БИПЭК АВТО,UAZ,Patriot,NaN,"13,800.00",0.00
38775,БИПЭК АВТО,Skoda,Kodiaq,NaN,"40,000.00",0.00


Максимальное расхождение цена × количество и сумма, USD: 0.0006999997422099113


## Скрытые проблемы форматов
Посмотрим на содержательные примеры, а не только на типы столбцов.

In [5]:
display(raw.loc[raw['Вид топлива'].isin(['2', '1,6']),
        ['Бренд', 'Модель', 'Вид топлива', 'Объём двиг, л,', 'Коробка передач', 'Тип привода']].drop_duplicates())
display(raw.loc[raw['Объём двиг, л,'].isin(['400 Л.С.', '88 KWH']),
        ['Бренд', 'Модель', 'Вид топлива', 'Объём двиг, л,']].drop_duplicates())
print('Компания — вариантов:', raw['Компания'].nunique())
print('Коробка передач — вариантов:', raw['Коробка передач'].nunique())
print('Тип привода — вариантов:', raw['Тип привода'].nunique())

,Бренд,Модель,Вид топлива,"Объём двиг, л,",Коробка передач,Тип привода
23723,Renault,Duster,2,MT,4WD,4WD
23724,Renault,Duster,2,AT,4WD,4WD
23728,Renault,Duster,"1,6",MT,4WD,4WD
23734,Renault,Duster,"1,6",AT,Передний,Передний
24997,Renault,Sandero,"1,6",0,0,0
24998,Renault,Sandero,"1,6",MT,Передний,Передний


,Бренд,Модель,Вид топлива,"Объём двиг, л,"
39544,Jaguar,I-Pace,Электро,400 Л.С.
39818,ANKAI,HFF6124G03EV3,Электричество,88 KWH


Компания — вариантов: 40
Коробка передач — вариантов: 83
Тип привода — вариантов: 26


## Вывод этапа и сверка с заданием
Файл содержит 39 966 строк, 25 полей и девять месяцев 2019 года; полностью пустых строк нет. Найдено 18 698 повторов сверх первого экземпляра. Есть 6 неизвестных количеств при нулевой сумме, один возврат и две крупные партии по 100 и 115 автомобилей. При известных количествах суммы согласуются с ценой с точностью округления.

**Выполнено:** загрузка с `sep=';'`, `decimal=','`; размер, типы, пропуски, дубликаты и числовые аномалии проверены. Следующий этап — тетрадь 02; все исходные сведения сохранены.